<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Background Subtraction</b></h1>
</div>

## Context
Neurointerventional fluoroscopy combines a largely static anatomical background with thin moving devices such as guidewires and microcatheters. These tools can be difficult to isolate because of low contrast, acquisition noise, residual anatomical structure, and their small spatial support.

The project uses the supplied fluoroscopy sequence and manual tool annotations to build and evaluate a reproducible background-subtraction workflow.


## Problem Statement
Develop a classical image-processing pipeline that automatically highlights moving neurointerventional tools while suppressing the mostly static anatomy.

The implementation must preserve one processing logic across the evaluation sequence, compare three segmentation strategies, and produce binary masks using the laboratory convention:

- **0 = tool**
- **1 = background**

Performance is evaluated against the supplied guidewire and microcatheter annotations.


## Inputs and Fixed Parameters

| Item | Value / convention |
|---|---|
| Frames | 201, 211, ..., 291 |
| Background frame | 201 |
| Evaluation frames | 211, 221, ..., 291 |
| Representative frame | 251 |
| Spatial Gaussian sigma | 1.0 |
| Preprocessing saturation | 0th–90th percentile |
| Gaussian HPF cutoff | 10.0 |
| Dilation radius | 2 |
| Residual saturation | 10th–100th percentile |
| Fixed threshold | 0.10 |
| Opening radius | 2 |
| GMM components | 2 |
| GMM random state | 0 |
| Required mask convention | 0 = tool, 1 = background |
| Primary metrics | SAD, MSE, PSNR |
| Supplementary metrics | Dice, IoU |


## 1. Reusable Processing Functions
Create the reusable functions needed by the project: intensity saturation, frame loading, ground-truth construction, Gaussian preprocessing, Gaussian spectral high-pass filtering, feature construction, three segmentation strategies, final morphology, quantitative metrics, and visualization overlays.

These helpers must be shared by all evaluated frames so that the processing logic remains identical throughout the sequence.


## 2. Data and Ground-Truth Validation
Before any processing, verify the complete dataset inventory.

For every frame identifier:
1. confirm the fluoroscopy image exists;
2. confirm guidewire and microcatheter annotations exist;
3. load the frame successfully;
4. verify that all frames have the same spatial shape.

The workflow must stop if the dataset is incomplete or geometrically inconsistent.


## 3. Static Background Reference
Use frame 201 as the static-background reference.

The frame is loaded, Gaussian-smoothed, and contrast-normalized with the same preprocessing function used later in the foreground pipeline. The processed image becomes the reference against which moving structures are detected.


## 4. Intermediate Processing Pipeline
Use frame 251 as a representative case and expose the intermediate processing chain:

1. input fluoroscopy frame;
2. Gaussian smoothing and percentile contrast normalization;
3. signed subtraction from the processed background;
4. centered Fourier transform;
5. Gaussian high-pass filtering;
6. morphological dilation;
7. percentile normalization of the residual feature.

The purpose is to verify that each required processing domain contributes to the final feature image.


## 5. Segmentation Strategy Generation
Generate three candidate segmentations from the same feature image:

- deterministic fixed threshold;
- Otsu thresholding;
- two-component EM/GMM classification.

Each candidate tool mask is refined with the same morphological opening and converted to the required binary convention.


## 6. Representative Strategy Comparison
Compare the three candidate masks on frame 251 using the manual annotation.

Display both the predicted mask and a semantic agreement overlay so that correct overlap, missed annotation, and false-positive regions can be inspected under identical visualization conditions.


## 7. Sequence-Level Evaluation
Apply the complete processing chain to every evaluation frame from 211 to 291.

For each frame and each segmentation strategy, compute SAD, MSE, PSNR, Dice, and IoU. Store the per-frame results in a structured table for sequence-level analysis.


## 8. Aggregate Metrics and Strategy Selection
Aggregate the per-frame results by segmentation strategy.

For each metric, compute the mean and standard deviation across the evaluation sequence. The retained strategy is selected by the minimum mean MSE, matching the executable code notebook.


## 9. Temporal Metric Curves
Plot SAD, MSE, and PSNR against frame number for each segmentation strategy.

The curves must expose temporal variation and make unstable frames or strategy-specific degradation visible.


## 10. Strategy Summary
Create a compact strategy-level comparison using the aggregate metrics.

The summary visualization must include mean MSE, mean PSNR, and mean Dice so that the retained strategy can be interpreted using both pixel-domain and overlap-oriented evidence.


## 11. Final Pipeline Assembly
Assemble the selected method into one reusable final-pipeline function.

Given a frame number, the function must load the frame and ground truth, build the feature image, apply the retained segmentation rule, finalize the binary mask, and return the data required for evaluation and visualization.


## 12. Final Guidance Gallery
Run the final pipeline over all evaluation frames and create the neurointervention guidance gallery.

The predicted tool region is overlaid in green on a contrast-enhanced grayscale fluoroscopy image so that the output can be interpreted as a guidance visualization rather than only as a binary mask.


## 13. Final Qualitative Validation
Select representative validation frames 211, 251, and 291.

For each one, present the final prediction, manual annotation, and semantic error overlay. The validation view must distinguish correct overlap from false-positive and missed-tool regions.


## 14. Export Results and Output Inventory
Export the detailed per-frame metrics and the aggregate strategy summary to CSV files.

Finally, enumerate the generated figures so that the notebook ends with an explicit output inventory and the project artifacts can be checked reproducibly.


## Completion Criterion
The project is complete when all 10 image/annotation triplets are validated, one consistent processing chain is applied to the nine evaluation frames, the three segmentation strategies are compared quantitatively, the retained strategy is assembled into the final inference function, guidance and validation figures are generated, and the metric CSV files are exported successfully.
